# Assignment 4 - Spark ML - Miguel Morales

## Learning Outcomes
In this assignment you will: 

-  Use ML piplenes
-  Improve a Random Forest model
-  Perform Hyperparameter tuning

#### Question 1:  (5 marks)

In our learning from this module, we have identified a fairly significant link by leveraging the ML pipeline, a more sophisticated model, and better hyperparameter tuning. However these results are still a bit disappointing. With that being said, we're working with very few features and we've likely made some assumptions that just aren't quite valid (like zip code shortening). Also, just because a rich zip code exists doesn't mean that the farmer's market would be held in that zip code too. In fact we might want to start looking at neighboring zip codes or doing some sort of distance measure to predict whether or not there exists a farmer's market in a certain mile radius from a wealthy zip code.

With that being said, we've got a lot of other potential features and plenty of other parameters to tune on our random forest so play around with the above pipeline and see if you can improve it further! Note: adding a feaure for the distance measure is just an example and not a mandatory change to improve the model's performance. We also aren't concerned about if the model's perforamnce is actually improved! We simply want to see if changes have been made to the code for possible improvements. 

Learn mode about the Farmers Markets dataset, here: https://catalog.data.gov/dataset/farmers-markets-directory-and-geographic-data (more info: Longitude and latitude, state, address, name, and zip code of Farmers Markets in the United States)
    
You may use the same classifier we built in this notebook.

In [ ]:
# To fix my setup I installed first Java 11 and then I installed Spark 3.4.1.
# winget install --id EclipseAdoptium.Temurin.11.JDK -e
# I set the environment variables for both Java and Spark, and then I was able to run Spark in standalone mode on my local machine.

# Set Java Home
#$jdk = Get-ChildItem "C:\Program Files\Eclipse Adoptium" -Directory |
#  Where-Object { $_.Name -like "jdk-11*" } |
#  Sort-Object Name -Descending |
#  Select-Object -First 1
#
#[Environment]::SetEnvironmentVariable("JAVA_HOME", $jdk.FullName, "User")
#$path = [Environment]::GetEnvironmentVariable("Path", "User")
#if ($path -notlike "*%JAVA_HOME%\bin*") {
#  [Environment]::SetEnvironmentVariable("Path", "$path;%JAVA_HOME%\bin", "User")
#}
#


# I also installed the PySpark library to use Spark with Python.
# I also replaced all the google colab specific code with the appropriate code for my local setup, 
# (for example the display function for the data frames).

In [ ]:
# One-time install
# %pip install -q pyspark==3.4.0 jupysql findspark

In [1]:
import os
import findspark
from pyspark.sql import SparkSession

# Use your installed Java 11 path
os.environ["JAVA_HOME"] = r"C:\Program Files\Eclipse Adoptium\jdk-11.0.30.7-hotspot"

findspark.init()

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("IoT_Data_Processing") \
    .config("spark.driver.memory", "12g") \
    .config("spark.sql.execution.arrow.pyspark.enabled", True) \
    .config("spark.sql.repl.eagerEval.enabled", True) \
    .getOrCreate()

# Verify the Spark version
print(f"Spark version: {spark.version}")

Spark version: 3.4.0


In [2]:
#%pip install -q grpcio
# %reload_ext sql

# CSV options
infer_schema = "false"
first_row_is_header = "false"
delimiter = ","

In [3]:
# Read The data
taxes2013 = (spark.read
    .option("header", "true")
    .csv("./2013_soi_zipcode_agi.csv"))

markets = (spark.read
  .option("header", "true")
  .csv("./market_data.csv"))

In [4]:
# Register spark SQL tables

taxes2013.createOrReplaceTempView("taxes2013")
markets.createOrReplaceTempView("markets")

In [5]:
taxes2013.schema

StructType([StructField('STATEFIPS', StringType(), True), StructField('STATE', StringType(), True), StructField('zipcode', StringType(), True), StructField('agi_stub', StringType(), True), StructField('N1', StringType(), True), StructField('MARS1', StringType(), True), StructField('MARS2', StringType(), True), StructField('MARS4', StringType(), True), StructField('PREP', StringType(), True), StructField('N2', StringType(), True), StructField('NUMDEP', StringType(), True), StructField('A00100', StringType(), True), StructField('N02650', StringType(), True), StructField('A02650', StringType(), True), StructField('N00200', StringType(), True), StructField('A00200', StringType(), True), StructField('N00300', StringType(), True), StructField('A00300', StringType(), True), StructField('N00600', StringType(), True), StructField('A00600', StringType(), True), StructField('N00650', StringType(), True), StructField('A00650', StringType(), True), StructField('N00700', StringType(), True), StructF

In [7]:
from pyspark.sql.types import IntegerType, DoubleType

In [8]:
cleaned_taxes = taxes2013.select('state', (taxes2013.zipcode / 10).cast(IntegerType()).alias('zipcode'), (taxes2013.MARS1).cast(IntegerType()).alias('single_returns'), (taxes2013.MARS2).cast(IntegerType()).alias('joint_returns'),(taxes2013.NUMDEP).cast(IntegerType()).alias('numdep'))
#df.select(concat($"firstname",$"lastname") as "full_name",$"salary" as "old_salary",$"salary"+5000 ali

#CREATE TABLE cleaned_taxes_ms AS
#SELECT state, int(zipcode / 10) as zipcode,
#  int(mars1) as single_returns,
#  int(mars2) as joint_returns,
#  int(numdep) as numdep,
#  double(A02650) as total_income_amount,
#  double(A00300) as taxable_interest_amount,
#  double(a01000) as net_capital_gains,
#  double(a00900) as biz_net_income
#FROM taxes2013

display(cleaned_taxes)

state,zipcode,single_returns,joint_returns,numdep
AL,0,488030,122290,571240
AL,0,195840,155230,383240
AL,0,72710,146880,189340
AL,0,24860,126480,134370
AL,0,16930,168170,177800
AL,0,3530,42190,48270
AL,3500,950,260,710
AL,3500,590,410,860
AL,3500,290,490,620
AL,3500,90,490,530


In [9]:
#sqlContext.cacheTable("cleaned_taxes")

# Convert back to a dataset from a table
#cleanedTaxes2 = spark.sql("SELECT * FROM cleaned_taxes")

summedTaxes = cleaned_taxes.groupBy("zipcode").sum() # because of AGI, where groups income groups are broken out

cleanedMarkets = (markets
  .selectExpr("*", "int(zip / 10) as zipcode")
  .groupBy("zipcode")
  .count()
  .selectExpr("double(count) as count", "zipcode as zip"))
#  selectExpr is short for Select Expression - equivalent to what we
#  might be doing in SQL SELECT expression

joined = (cleanedMarkets.join(summedTaxes, cleanedMarkets.zip == summedTaxes.zipcode, "outer"))

In [10]:
display(summedTaxes)
# also without display works nicely because this was enable: .config("spark.sql.repl.eagerEval.enabled", True)
# summnedTaxes 

zipcode,sum(zipcode),sum(single_returns),sum(joint_returns),sum(numdep)
8592,257760,2390,2950,4430
7240,86880,12980,12170,22710
3561,149562,9880,12310,19700
3566,42792,4700,4710,7280
8501,408048,52230,24880,86240
3631,174288,3160,3780,5670
3691,88584,620,600,1570
8534,307224,17600,15670,32930
8640,259200,17580,17680,22360
7206,345888,2250,2840,4080


In [12]:
display(cleanedMarkets)

count,zip
5.0,4900
2.0,7240
8.0,4818
1.0,9852
2.0,5300
5.0,2122
2.0,9900
1.0,8592
1.0,1580
1.0,3175


In [13]:
display(joined)

count,zip,zipcode,sum(zipcode),sum(single_returns),sum(joint_returns),sum(numdep)
1009.0,null,null,null,null,null,null
1.0,0,0,0,66430180,52885400,96500590
1.0,3,null,null,null,null,null
4.0,60,null,null,null,null,null
1.0,61,null,null,null,null,null
2.0,62,null,null,null,null,null
1.0,63,null,null,null,null,null
1.0,65,null,null,null,null,null
4.0,66,null,null,null,null,null
4.0,67,null,null,null,null,null


In [15]:
#Dealing with nulls
#joined.na.fill(0).show()
prepped = joined.na.fill(0)
display(prepped)

count,zip,zipcode,sum(zipcode),sum(single_returns),sum(joint_returns),sum(numdep)
1009.0,0,0,0,0,0,0
1.0,0,0,0,66430180,52885400,96500590
1.0,3,0,0,0,0,0
4.0,60,0,0,0,0,0
1.0,61,0,0,0,0,0
2.0,62,0,0,0,0,0
1.0,63,0,0,0,0,0
1.0,65,0,0,0,0,0
4.0,66,0,0,0,0,0
4.0,67,0,0,0,0,0


Now that all of our data is prepped. We're going to have to put all of it into one column of a vector type for Spark MLLib. This makes it easy to embed a prediction right in a DataFrame and also makes it very clear as to what is getting passed into the model and what isn't without having to convert it to a numpy array or specify an R formula. This also makes it easy to incrementally add new features, simply by adding to the vector. In the below case rather than specifically adding them in.

In [16]:
nonFeatureCols = ["zip", "zipcode", "count"]
featureCols = [item for item in prepped.columns if item not in nonFeatureCols]

In [17]:
# VectorAssembler Assembles all of these columns into one single vector. To do this, set the input columns and output column. Then that assembler will be used to transform the prepped data to the final dataset.
from pyspark.ml.feature import VectorAssembler

assembler = (VectorAssembler()
  .setInputCols(featureCols)
  .setOutputCol("features"))

finalPrep = assembler.transform(prepped)

Now split the dataset 70-30 for training and testing purposes.A validation set can be created as well, we are omitting it here. It's worth noting that MLLib also supports performing hyperparameter tuning with cross validation and pipelines. All this can be found in [the Databrick's Guide](https://docs.databricks.com).

In [18]:
training, test = finalPrep.randomSplit([0.7, 0.3])

#  Going to cache the data to make sure things stay snappy!
training.cache()
test.cache()

print(training.count())
print(test.count())

4155
1647


In [19]:
display(training)

count,zip,zipcode,sum(zipcode),sum(single_returns),sum(joint_returns),sum(numdep),features
0.0,0,833,9996,14000,9490,19940,"[9996.0,14000.0,9..."
0.0,0,2366,99372,31870,23150,43120,"[99372.0,31870.0,..."
0.0,0,2659,31908,370,450,570,"[31908.0,370.0,45..."
0.0,0,2866,85980,1320,1720,2230,"[85980.0,1320.0,1..."
0.0,0,5803,243726,690,720,860,"[243726.0,690.0,7..."
0.0,0,6654,359316,3180,4190,4900,"[359316.0,3180.0,..."
0.0,0,7880,47280,3370,2800,7230,"[47280.0,3370.0,2..."
0.0,0,7982,47892,820,740,2050,"[47892.0,820.0,74..."
0.0,0,7993,287748,46200,43920,107230,"[287748.0,46200.0..."
1.0,1580,1580,9480,4600,3940,4650,"[9480.0,4600.0,39..."


# Apache Spark MLLib

At a high level, we're going to create an instance of a `regressor` or `classifier`, that in turn will then be trained and return a `Model` type. Whenever you access Spark MLLib you should be sure to import/train on the name of the algorithm you want as opposed to the `Model` type. For example:

You should import:

`org.apache.spark.ml.regression.LinearRegression`

as opposed to:

`org.apache.spark.ml.regression.LinearRegressionModel`

In the below example, we're going to use linear regression.

The linear regression that is available in Spark MLLib supports an elastic net parameter allowing you to set a threshold of how much you would like to mix l1 and l2 regularization, for [more information on Elastic net regularization see Wikipedia](https://en.wikipedia.org/wiki/Elastic_net_regularization).

As we saw above, we had to perform some preparation of the data before inputting it into the model. We've got to do the same with the model itself. We'll set our hyper parameters, print them out and then finally we can train it! The `explainParams` is a great way to ensure that you're taking advantage of all the different hyperparameters that you have available.

In [23]:
from pyspark.ml.regression import LinearRegression

lrModel = (LinearRegression()
  .setLabelCol("count")
  .setFeaturesCol("features")
  .setElasticNetParam(0.5))

print("Printing out the model Parameters:")
print("-"*100)
print(lrModel.explainParams())
print("-"*100)

Printing out the model Parameters:
----------------------------------------------------------------------------------------------------
aggregationDepth: suggested depth for treeAggregate (>= 2). (default: 2)
elasticNetParam: the ElasticNet mixing parameter, in range [0, 1]. For alpha = 0, the penalty is an L2 penalty. For alpha = 1, it is an L1 penalty. (default: 0.0, current: 0.5)
epsilon: The shape parameter to control the amount of robustness. Must be > 1.0. Only valid when loss is huber (default: 1.35)
featuresCol: features column name. (default: features, current: features)
fitIntercept: whether to fit an intercept term. (default: True)
labelCol: label column name. (default: label, current: count)
loss: The loss function to be optimized. Supported options: squaredError, huber. (default: squaredError)
maxBlockSizeInMB: maximum memory in MB for stacking input data into blocks. Data is stacked within partitions. If more than remaining data size in a partition then it is adjusted to 

Now finally we can go about fitting our model! You'll see that we're going to do this in a series of steps. First we'll fit it, then we'll use it to make predictions via the `transform` method. This is the same way you would make predictions with your model in the future however in this case we're using it to evaluate how our model is doing. We'll be using regression metrics to get some idea of how our model is performing, we'll then print out those values to be able to evaluate how it performs.

In [24]:
from pyspark.mllib.evaluation import RegressionMetrics
lrFitted = lrModel.fit(training)

Now you'll see that since we're working with exact numbers (you can't have 1/2 a farmer's market for example), I'm going to check equality by first rounding the value to the nearest digital value.

In [25]:
holdout = (lrFitted
  .transform(test)
  .selectExpr("prediction as raw_prediction",
    "double(round(prediction)) as prediction",
    "count",
    """CASE double(round(prediction)) = count
  WHEN true then 1
  ELSE 0
END as equal"""))

display(holdout)

raw_prediction,prediction,count,equal
1.1346111626756725,1.0,0.0,0
1.5605023547531751,2.0,0.0,0
1.2028368519311556,1.0,1.0,1
1.0923089201862677,1.0,1.0,1
1.3650042312940722,1.0,1.0,1
1.350470426312096,1.0,1.0,1
1.2453875614873169,1.0,1.0,1
1.3244001493069941,1.0,1.0,1
1.1143701841130549,1.0,2.0,0
1.072210458537938,1.0,2.0,0


Now let's see what proportion was exactly correct.

In [26]:
display(holdout.selectExpr("sum(equal)/sum(1)"))

(sum(equal) / sum(1))
0.30115361262902246


In [28]:
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.sql import functions as F

# The original code (commented out below) used pyspark.mllib.evaluation.RegressionMetrics,
# which is the older RDD-based API. It required converting the DataFrame to an RDD using
# .rdd.map. On Windows, this causes serialization issues.
#
# The fix uses pyspark.ml.evaluation.RegressionEvaluator (the newer DataFrame-based API)
# which keeps all computation inside the JVM using the Catalyst optimizer.

#rm = RegressionMetrics(holdout.select("prediction", "count").rdd.map(lambda x:  (x[0], x[1])))

#print("MSE: ", rm.meanSquaredError)
#print("MAE: ", rm.meanAbsoluteError)
#print("RMSE Squared: ", rm.rootMeanSquaredError)
#print("R Squared: ", rm.r2)
#print("Explained Variance: ", rm.explainedVariance, "\n")

# Keep everything as DataFrame (no RDD/lambda conversion)
eval_df = holdout.select(
	F.col("prediction").cast("double").alias("prediction"),
	F.col("count").cast("double").alias("label")
).cache()

evaluator = RegressionEvaluator(labelCol="label", predictionCol="prediction")

mse = evaluator.setMetricName("mse").evaluate(eval_df)
mae = evaluator.setMetricName("mae").evaluate(eval_df)
rmse = evaluator.setMetricName("rmse").evaluate(eval_df)
r2 = evaluator.setMetricName("r2").evaluate(eval_df)

# Explained variance: 1 - Var(y - yhat) / Var(y)
stats = eval_df.select((F.col("label") - F.col("prediction")).alias("residual")).agg(
	F.var_pop("residual").alias("residual_var")
).crossJoin(
	eval_df.agg(F.var_pop("label").alias("label_var"))
).first()

residual_var = 0.0 if stats["residual_var"] is None else stats["residual_var"]
label_var = stats["label_var"]
explained_variance = 1.0 if (label_var is None or label_var == 0) else 1.0 - (residual_var / label_var)

print("MSE: ", mse)
print("MAE: ", mae)
print("RMSE: ", rmse)
print("R Squared: ", r2)
print("Explained Variance: ", explained_variance, "\n")

MSE:  618.9119611414693
MAE:  1.5907710989678203
RMSE:  24.87794125609009
R Squared:  -0.0004363084308027787
Explained Variance:  0.0004681628601211907 



These results appear to be sub-optimal, so let's try exploring another way to train the model. Rather than training on a single model with hard-coded parameters, let's train using a [pipeline](http://spark.apache.org/docs/latest/api/scala/index.html#org.apache.spark.ml.Pipeline).

A pipeline is going to give us some nice benefits in that it will allow us to use a couple of transformations we need in order to transform our raw data into the prepared data for the model but also it provides a simple, straightforward way to try out a lot of different combinations of parameters. This is a process called [hyperparameter tuning](https://en.wikipedia.org/wiki/Hyperparameter_optimization) or grid search. To review, grid search is where you set up the exact parameters that you would like to test and MLLib will automatically create all the necessary combinations of these to test.

For example, below we'll set `numTrees` to 20 and 60 and `maxDepth` to 5 and 10. The parameter grid builder will automatically construct all the combinations of these two variable (along with the other ones that we might specify too). Additionally we're also going to use [cross validation](https://en.wikipedia.org/wiki/Cross-validation_(statistics)) to tune our hyperparameters, this will allow us to attempt to try to control [overfitting](https://en.wikipedia.org/wiki/Overfitting) of our model.

Lastly we'll need to set up a [Regression Evaluator](http://spark.apache.org/docs/latest/api/scala/index.html#org.apache.spark.ml.evaluation.RegressionEvaluator) that will evaluate the models that we choose based on some metric (the default is RMSE). The key take away is that the pipeline will automatically optimize for our given metric choice by exploring the parameter grid that we set up rather than us having to do it manually like we would have had to do above.

Now we can go about training our random forest!

*note: this might take a little while because of the number of combinations that we're trying and limitations in workers available.*

In [29]:
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from pyspark.ml.evaluation import RegressionEvaluator

from pyspark.ml import Pipeline

rfModel = (RandomForestRegressor()
  .setLabelCol("count")
  .setFeaturesCol("features"))

paramGrid = (ParamGridBuilder()
  .addGrid(rfModel.maxDepth, [5])
  .addGrid(rfModel.numTrees, [20])
  .build())
# Note, that this parameter grid will take a long time
# to run in the community edition due to limited number
# of workers available! Be patient for it to run!
# If you want it to run faster, remove some of
# the above parameters and it'll speed right up!

stages = [rfModel]

pipeline = Pipeline().setStages(stages)

cv = (CrossValidator() # you can feel free to change the number of folds used in cross validation as well
  .setEstimator(pipeline) # the estimator can also just be an individual model rather than a pipeline
  .setEstimatorParamMaps(paramGrid)
  .setEvaluator(RegressionEvaluator().setLabelCol("count")))

pipelineFitted = cv.fit(training)

Now we've trained our model! Let's take a look at which version performed best!

In [30]:
print("The Best Parameters:\n--------------------")
print(pipelineFitted.bestModel.stages[0])
pipelineFitted.bestModel.stages[0].extractParamMap()

The Best Parameters:
--------------------
RandomForestRegressionModel: uid=RandomForestRegressor_01af196454e1, numTrees=20, numFeatures=4


{Param(parent='RandomForestRegressor_01af196454e1', name='bootstrap', doc='Whether bootstrap samples are used when building trees.'): True,
 Param(parent='RandomForestRegressor_01af196454e1', name='cacheNodeIds', doc='If false, the algorithm will pass trees to executors to match instances with nodes. If true, the algorithm will cache node IDs for each instance. Caching can speed up training of deeper trees. Users can set how often should the cache be checkpointed or disable it by setting checkpointInterval.'): False,
 Param(parent='RandomForestRegressor_01af196454e1', name='checkpointInterval', doc='set checkpoint interval (>= 1) or disable checkpoint (-1). E.g. 10 means that the cache will get checkpointed every 10 iterations. Note: this setting will be ignored if the checkpoint directory is not set in the SparkContext.'): 10,
 Param(parent='RandomForestRegressor_01af196454e1', name='featureSubsetStrategy', doc="The number of features to consider for splits at each tree node. Supporte

As well as our regression metrics on the test set.

In [31]:
pipelineFitted.bestModel

PipelineModel_d42bc7f67768

In [32]:
holdout2 = (pipelineFitted.bestModel
  .transform(test)
  .selectExpr("prediction as raw_prediction",
    "double(round(prediction)) as prediction",
    "count",
    """CASE double(round(prediction)) = count
  WHEN true then 1
  ELSE 0
END as equal"""))

display(holdout2)

raw_prediction,prediction,count,equal
0.35020458563528273,0.0,0.0,1
0.419006831443938,0.0,0.0,1
1.1066842856913506,1.0,1.0,1
0.7417817468658423,1.0,1.0,1
0.8000299643842734,1.0,1.0,1
0.27774943134281377,0.0,1.0,0
1.2478016943219574,1.0,1.0,1
1.5843902933013077,2.0,1.0,0
1.104717860861427,1.0,2.0,0
1.6429937202098646,2.0,2.0,1


In [34]:
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.sql import functions as F

# Same fix as applied to the first model evaluation: replaced the older RDD-based
# RegressionMetrics (pyspark.mllib) with the DataFrame-based RegressionEvaluator
# (pyspark.ml) to avoid Python lambda serialization failures on Windows.

#rm2 = RegressionMetrics(holdout2.select("prediction", "count").rdd.map(lambda x:  (x[0], x[1])))

#print("MSE: ", rm2.meanSquaredError)
#print("MAE: ", rm2.meanAbsoluteError)
#print("RMSE Squared: ", rm2.rootMeanSquaredError)
#print("R Squared: ", rm2.r2)
#print("Explained Variance: ", rm2.explainedVariance, "\n")

# Keep everything as DataFrame (no RDD/lambda conversion)
eval_df2 = holdout2.select(
    F.col("prediction").cast("double").alias("prediction"),
    F.col("count").cast("double").alias("label")
).cache()

evaluator2 = RegressionEvaluator(labelCol="label", predictionCol="prediction")

mse2 = evaluator2.setMetricName("mse").evaluate(eval_df2)
mae2 = evaluator2.setMetricName("mae").evaluate(eval_df2)
rmse2 = evaluator2.setMetricName("rmse").evaluate(eval_df2)
r2_2 = evaluator2.setMetricName("r2").evaluate(eval_df2)

# Explained variance: 1 - Var(y - yhat) / Var(y)
stats2 = eval_df2.select((F.col("label") - F.col("prediction")).alias("residual")).agg(
    F.var_pop("residual").alias("residual_var")
).crossJoin(
    eval_df2.agg(F.var_pop("label").alias("label_var"))
).first()

residual_var2 = 0.0 if stats2["residual_var"] is None else stats2["residual_var"]
label_var2 = stats2["label_var"]
explained_variance2 = 1.0 if (label_var2 is None or label_var2 == 0) else 1.0 - (residual_var2 / label_var2)

print("MSE: ", mse2)
print("MAE: ", mae2)
print("RMSE: ", rmse2)
print("R Squared: ", r2_2)
print("Explained Variance: ", explained_variance2, "\n")


MSE:  618.6527018822101
MAE:  1.514875531268974
RMSE:  24.872730085019015
R Squared:  -1.7230447915794755e-05
Explained Variance:  0.000533095697320296 



Finally we'll see an improvement in our "exactly right" proportion as well!

In [35]:
display(holdout2.selectExpr("sum(equal)/sum(1)"))

(sum(equal) / sum(1))
0.36429872495446264


## Question 1 — Improving the Random Forest Model

Applying some concepts that I learned in my previous course (Machine Learning):

### Rational of approached followed to improve models

The existing Random Forest uses a `paramGrid` with only one value per parameter (`maxDepth=[5]`, `numTrees=[20]`), which means `CrossValidator` is fitting a single fixed configuration. In other words, if I understand correctly, there is no actual hyperparameter search happening. Thus, 2 improvements were explored here:

#### 1. Hyperparameter Tuning (expanded `paramGrid`)
The grid is expanded to test 4 combinations across 2 hyperparameters:
- `maxDepth`: `[5, 10]` — this controls how deep each tree can grow, deeper trees can capture more complex patterns (but we need to be careful because they can overfit)
- `numTrees`: `[20, 50]` — more trees generally improve stability and accuracy (at the cost of compute time)

`CrossValidator` (3-fold) will now actually compare these combinations and select the best.

#### 2. Feature Scaling
A `StandardScaler` stage is added to the pipeline before the Random Forest. In the course we saw that tree-based models are not strictly sensitive to feature scale, but standardizing ensures that the features are on a consistent scale, which helps `CrossValidator` compare models fairly (good general practice).

---

After this change, the pipeline becomes: **VectorAssembler → StandardScaler → RandomForestRegressor**

---

Note: the cell below will take approximately ~12 min to run

In [37]:
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml import Pipeline
from pyspark.sql import functions as F

# ---------------------------------------------------------------------------
# Question 1 — Improved Random Forest Model
#
# Two improvements over the original model (pipelineFitted):
#   1. Expanded paramGrid: the original tested only 1 combination (maxDepth=5,
#      numTrees=20), so CrossValidator was not doing real hyperparameter tuning.
#      Here we test 4 combinations (2 depths x 2 tree counts).
#   2. StandardScaler added as a pipeline stage before the RF model, ensuring
#      features are on a consistent scale — a good general practice.
#
# Pipeline: VectorAssembler → StandardScaler → RandomForestRegressor
# ---------------------------------------------------------------------------

# Step 1: Assemble raw features into a vector (same featureCols as before)
assembler2 = VectorAssembler(inputCols=featureCols, outputCol="rawFeatures2")

# Step 2: Scale the assembled features
# Use a new output column name to avoid collision with existing 'features'
scaler = StandardScaler(inputCol="rawFeatures2", outputCol="scaledFeatures2", withMean=True, withStd=True)

# Step 3: Random Forest Regressor
rf2 = RandomForestRegressor(labelCol="count", featuresCol="scaledFeatures2")

# Step 4: Build pipeline with all three stages
pipeline2 = Pipeline(stages=[assembler2, scaler, rf2])

# Step 5: Expanded parameter grid — 4 combinations vs the original 1
paramGrid2 = (ParamGridBuilder()
    .addGrid(rf2.maxDepth, [5, 10])
    .addGrid(rf2.numTrees, [20, 50])
    .build())

# Step 6: CrossValidator with 3-fold CV (same as original)
cv2 = (CrossValidator()
    .setEstimator(pipeline2)
    .setEstimatorParamMaps(paramGrid2)
    .setEvaluator(RegressionEvaluator(labelCol="count"))
    .setNumFolds(3))

# Step 7: Fit on training data (uses the same training/test split as before)
print("Training improved model — testing", len(paramGrid2), "hyperparameter combinations x 3 folds...")
pipelineFitted2 = cv2.fit(training)
print("Done.\n")

# Step 8: Show best parameters selected by CrossValidator
print("Best Parameters selected by CrossValidator:")
print("-" * 60)
bestRF2 = pipelineFitted2.bestModel.stages[-1]  # last stage is the RF
print(f"  numTrees : {bestRF2.getNumTrees}")
print(f"  maxDepth : {bestRF2.getOrDefault('maxDepth')}")
print("-" * 60)

# Step 9: Generate predictions on test set
holdout3 = (pipelineFitted2.bestModel
    .transform(test)
    .selectExpr(
        "prediction as raw_prediction",
        "double(round(prediction)) as prediction",
        "count",
        """CASE double(round(prediction)) = count
  WHEN true then 1
  ELSE 0
END as equal"""))

# Step 10: Evaluate using DataFrame-based RegressionEvaluator (Windows-safe)
eval_df3 = holdout3.select(
    F.col("prediction").cast("double").alias("prediction"),
    F.col("count").cast("double").alias("label")
).cache()

evaluator3 = RegressionEvaluator(labelCol="label", predictionCol="prediction")

mse3  = evaluator3.setMetricName("mse").evaluate(eval_df3)
mae3  = evaluator3.setMetricName("mae").evaluate(eval_df3)
rmse3 = evaluator3.setMetricName("rmse").evaluate(eval_df3)
r2_3  = evaluator3.setMetricName("r2").evaluate(eval_df3)

# Explained variance: 1 - Var(residuals) / Var(labels)
stats3 = eval_df3.select((F.col("label") - F.col("prediction")).alias("residual")).agg(
    F.var_pop("residual").alias("residual_var")
).crossJoin(
    eval_df3.agg(F.var_pop("label").alias("label_var"))
).first()

residual_var3 = 0.0 if stats3["residual_var"] is None else stats3["residual_var"]
label_var3    = stats3["label_var"]
explained_variance3 = 1.0 if (label_var3 is None or label_var3 == 0) else 1.0 - (residual_var3 / label_var3)

exact_proportion3 = holdout3.selectExpr("sum(equal)/sum(1)").first()[0]

# Step 11: Print results and comparison with the original two models
print("\n" + "=" * 65)
print("MODEL COMPARISON")
print("=" * 65)
print(f"{'Metric':<25} {'Model 1 (LR)':>12} {'Model 2 (RF)':>12} {'Model 3 (RF+)':>13}")
print("-" * 65)
print(f"{'MSE':<25} {mse:>12.4f} {mse2:>12.4f} {mse3:>13.4f}")
print(f"{'MAE':<25} {mae:>12.4f} {mae2:>12.4f} {mae3:>13.4f}")
print(f"{'RMSE':<25} {rmse:>12.4f} {rmse2:>12.4f} {rmse3:>13.4f}")
print(f"{'R Squared':<25} {r2:>12.4f} {r2_2:>12.4f} {r2_3:>13.4f}")
print(f"{'Explained Variance':<25} {explained_variance:>12.4f} {explained_variance2:>12.4f} {explained_variance3:>13.4f}")
print("=" * 65)
print(f"\nExact proportion correct:  Model 3 (RF+) = {exact_proportion3:.4f}")


Training improved model — testing 4 hyperparameter combinations x 3 folds...
Done.

Best Parameters selected by CrossValidator:
------------------------------------------------------------
  numTrees : 50
  maxDepth : 5
------------------------------------------------------------

MODEL COMPARISON
Metric                    Model 1 (LR) Model 2 (RF) Model 3 (RF+)
-----------------------------------------------------------------
MSE                           618.9120     618.6527      618.6545
MAE                             1.5908       1.5149        1.5131
RMSE                           24.8779      24.8727       24.8728
R Squared                      -0.0004      -0.0000       -0.0000
Explained Variance              0.0005       0.0005        0.0005

Exact proportion correct:  Model 3 (RF+) = 0.3649


#### Question 2 ( 7 marks)


Using the Apache Spark ML pipeline, build a model to predict the price of a diamond based on the available features. Note: If you receive an R_Squared value that is negative, that is okay. This may occur due to the low sample size of the data.

Read from the following notebook for details about dataset.

https://databricks-prod-cloudfront.cloud.databricks.com/public/4027ec902e239c93eaaa8714f173bcfc/5915990090493625/4396972618536508/6085673883631125/latest.html

From the website:

The dataset diamonds (a data frame with 53940 rows and 10 variables) contains the prices and other attributes of almost 54,000 diamonds. The variables are as follows:

- price
price in US dollars ($326–$18,823)

- carat
weight of the diamond (0.2–5.01)

- cut
quality of the cut (Fair, Good, Very Good, Premium, Ideal)

- color
diamond colour, from D (best) to J (worst)

- clarity
a measurement of how clear the diamond is (I1 (worst), SI2, SI1, VS2, VS1, VVS2, VVS1, IF (best))

- x
length in mm (0–10.74)

- y
width in mm (0–58.9)

- z
depth in mm (0–31.8)

- depth
total depth percentage = z / mean(x, y) = 2 * z / (x + y) (43–79)

- table
width of top of diamond relative to widest point (43–95)

### Approach

The diamond dataset has 10 columns: 6 numeric (carat, depth, table, x, y, z), 3 categorical (cut, color, clarity), and the target (price).

Some data prep is needed, since as we saw in the course, Spark ML models require numeric types! The categorical features need to be indexed, and all features need to be assembled into a single vector column for the model.
**Pipeline stages:**
1. **Cast types** — CSV columns are all strings; cast numerics to double, and price to double
2. **StringIndexer** (×3) — Encode cut, color, and clarity as numeric indices.
3. **VectorAssembler** — Combine the 6 numeric + 3 indexed columns into a single features vector (9 features)
4. **RandomForestRegressor** — Predicts price from the assembled features
5. **CrossValidator** — 3-fold CV with a small param grid to select the best configuration


In [ ]:
# ---------------------------------------------------------------------------
# Question 2 — Diamond model
# ---------------------------------------------------------------------------


# Read The data
diamonds = (spark.read
    .option("header", "true")
    .csv("./diamonds.csv"))

In [39]:
from pyspark.sql.types import DoubleType
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml import Pipeline
from pyspark.sql import functions as F

# ---------------------------------------------------------------------------
# Step 1: Cast all numeric columns from string to double
# ---------------------------------------------------------------------------
numeric_cols = ["carat", "depth", "table", "x", "y", "z", "price"]
diamonds_typed = diamonds
for col_name in numeric_cols:
    diamonds_typed = diamonds_typed.withColumn(col_name, F.col(col_name).cast(DoubleType()))

print(f"Dataset size: {diamonds_typed.count()} rows")
diamonds_typed.printSchema()

# ---------------------------------------------------------------------------
# Step 2: StringIndexer for each categorical column
# ---------------------------------------------------------------------------
indexer_cut     = StringIndexer(inputCol="cut",     outputCol="cut_index")
indexer_color   = StringIndexer(inputCol="color",   outputCol="color_index")
indexer_clarity = StringIndexer(inputCol="clarity", outputCol="clarity_index")

# ---------------------------------------------------------------------------
# Step 3: VectorAssembler — combine 6 numeric + 3 indexed cols → features
# ---------------------------------------------------------------------------
feature_cols = ["carat", "depth", "table", "x", "y", "z",
                "cut_index", "color_index", "clarity_index"]

assembler_d = VectorAssembler(inputCols=feature_cols, outputCol="features")

# ---------------------------------------------------------------------------
# Step 4: RandomForestRegressor
# ---------------------------------------------------------------------------
rf_d = RandomForestRegressor(labelCol="price", featuresCol="features")

# ---------------------------------------------------------------------------
# Step 5: Build the full pipeline
# ---------------------------------------------------------------------------
pipeline_d = Pipeline(stages=[indexer_cut, indexer_color, indexer_clarity, assembler_d, rf_d])

# ---------------------------------------------------------------------------
# Step 6: Param grid — 4 combinations (same grid pattern as Q1)
# ---------------------------------------------------------------------------
paramGrid_d = (ParamGridBuilder()
    .addGrid(rf_d.maxDepth, [5, 10])
    .addGrid(rf_d.numTrees, [20, 50])
    .build())

# ---------------------------------------------------------------------------
# Step 7: CrossValidator (3-fold)
# ---------------------------------------------------------------------------
cv_d = (CrossValidator()
    .setEstimator(pipeline_d)
    .setEstimatorParamMaps(paramGrid_d)
    .setEvaluator(RegressionEvaluator(labelCol="price"))
    .setNumFolds(3))

# ---------------------------------------------------------------------------
# Step 8: Train / test split (70-30) and fit
# ---------------------------------------------------------------------------
training_d, test_d = diamonds_typed.randomSplit([0.7, 0.3], seed=42)
training_d.cache()
test_d.cache()

print(f"Training rows: {training_d.count()}")
print(f"Test rows:     {test_d.count()}")
print(f"\nTraining model — {len(paramGrid_d)} param combinations x 3 folds...")

pipelineFitted_d = cv_d.fit(training_d)
print("Done.\n")

# ---------------------------------------------------------------------------
# Step 9: Best model parameters
# ---------------------------------------------------------------------------
bestRF_d = pipelineFitted_d.bestModel.stages[-1]
print("Best Parameters selected by CrossValidator:")
print("-" * 50)
print(f"  numTrees : {bestRF_d.getNumTrees}")
print(f"  maxDepth : {bestRF_d.getOrDefault('maxDepth')}")
print("-" * 50)

# ---------------------------------------------------------------------------
# Step 10: Predictions on test set
# ---------------------------------------------------------------------------
predictions_d = pipelineFitted_d.bestModel.transform(test_d)
display(predictions_d.select("carat", "cut", "color", "clarity", "price", "prediction").limit(20))

# ---------------------------------------------------------------------------
# Step 11: Evaluate (same DataFrame-based approach as Q1, Windows-safe)
# ---------------------------------------------------------------------------
eval_df_d = predictions_d.select(
    F.col("prediction").cast("double").alias("prediction"),
    F.col("price").cast("double").alias("label")
).cache()

evaluator_d = RegressionEvaluator(labelCol="label", predictionCol="prediction")

mse_d  = evaluator_d.setMetricName("mse").evaluate(eval_df_d)
mae_d  = evaluator_d.setMetricName("mae").evaluate(eval_df_d)
rmse_d = evaluator_d.setMetricName("rmse").evaluate(eval_df_d)
r2_d   = evaluator_d.setMetricName("r2").evaluate(eval_df_d)

# Explained variance
stats_d = eval_df_d.select((F.col("label") - F.col("prediction")).alias("residual")).agg(
    F.var_pop("residual").alias("residual_var")
).crossJoin(
    eval_df_d.agg(F.var_pop("label").alias("label_var"))
).first()

residual_var_d = 0.0 if stats_d["residual_var"] is None else stats_d["residual_var"]
label_var_d    = stats_d["label_var"]
explained_variance_d = 1.0 if (label_var_d is None or label_var_d == 0) else 1.0 - (residual_var_d / label_var_d)

print("\n" + "=" * 55)
print("DIAMOND PRICE PREDICTION — MODEL RESULTS")
print("=" * 55)
print(f"  MSE:                {mse_d:>14.4f}")
print(f"  MAE:                {mae_d:>14.4f}")
print(f"  RMSE:               {rmse_d:>14.4f}")
print(f"  R Squared:          {r2_d:>14.4f}")
print(f"  Explained Variance: {explained_variance_d:>14.4f}")
print("=" * 55)


Dataset size: 53940 rows
root
 |-- _c0: string (nullable = true)
 |-- carat: double (nullable = true)
 |-- cut: string (nullable = true)
 |-- color: string (nullable = true)
 |-- clarity: string (nullable = true)
 |-- depth: double (nullable = true)
 |-- table: double (nullable = true)
 |-- price: double (nullable = true)
 |-- x: double (nullable = true)
 |-- y: double (nullable = true)
 |-- z: double (nullable = true)

Training rows: 37764
Test rows:     16176

Training model — 4 param combinations x 3 folds...
Done.

Best Parameters selected by CrossValidator:
--------------------------------------------------
  numTrees : 50
  maxDepth : 10
--------------------------------------------------


carat,cut,color,clarity,price,prediction
0.8,Premium,H,SI1,2760.0,2518.0011128213728
1.01,Very Good,H,SI1,4705.0,4769.912824410391
1.22,Fair,F,SI2,4705.0,5008.052604165118
0.91,Ideal,E,SI1,4706.0,4493.149945959873
0.9,Very Good,E,VS2,4707.0,4519.7988304741175
0.75,Ideal,D,SI1,2898.0,2997.12913089258
1.01,Very Good,G,SI1,4707.0,4939.9934278483215
1.01,Premium,H,SI1,4707.0,4694.799012147221
1.02,Ideal,G,SI1,4708.0,5149.092970950112
0.9,Very Good,E,VS2,4709.0,4442.42889677042



DIAMOND PRICE PREDICTION — MODEL RESULTS
  MSE:                   435059.6124
  MAE:                      348.9245
  RMSE:                     659.5905
  R Squared:                  0.9728
  Explained Variance:         0.9728
